<a href="https://colab.research.google.com/github/HHHOOOXX/codeit-toy-project/blob/main/NH%ED%88%AC%EC%9E%90%EC%A6%9D%EA%B6%8C%20%EB%B9%85%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EA%B2%BD%EC%A7%84%EB%8C%80%ED%9A%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 대형투자자 기관 데이터 크롤링
- dataroma에 있는 Superinvestor 모두에 대해 진행
- 각 기업마다의 activity 부분
- 각 기업마다의 현재 보유종목(holdings) 부분
- 2023년도 1분기부터 현재까지로 기간 설정

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
# Dataroma은 User-Agent값을 요구하므로 미리 header값 작성해두기
header = {'User-Agent':'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/12.0.2 Safari/605.1.15'}

## 1. dataroma에 있는 Superinvestor 리스트 가져오기

In [ ]:
# dataroma에 있는 Superinvestor 리스트 가져오기

import requests
from bs4 import BeautifulSoup
import re

# URL 설정
url = 'https://www.dataroma.com/m/home.php'

# GET 요청을 통해 데이터 가져오기
response = requests.get(url, headers=header)
if response.status_code == 200:
    # HTML 파싱
    soup = BeautifulSoup(response.text, 'html.parser')

    # 'href' 속성을 가진 모든 'a' 태그 찾기
    links = soup.find_all('a', href=True)

    # 'href'의 특정 부분을 찾기 위한 정규식
    pattern = re.compile(r'/m/holdings.php\?m=(\w+)')

    # 일치하는 식별자를 저장할 리스트
    identifiers = []

    # 각 링크에서 식별자 추출
    for link in links:
        match = pattern.search(link['href'])
        if match:
            identifiers.append(match.group(1))

    # 찾은 식별자들 출력
    print("찾은 식별자들:", identifiers)
else:
    print("웹페이지를 가져오는 데 실패했습니다. 상태 코드:", response.status_code)


찾은 식별자들: ['hcmax', 'GA', 'MVALX', 'oaklx', 'LLPFX', 'SEQUX', 'MP', 'GFT', 'vg', 'VFC', 'LPC', 'VA', 'HC', 'AIM', 'SA', 'mc', 'tp', 'ic', 'PC', 'TF', 'GLRE', 'fairx', 'CCM', 'BRK', 'ENG', 'AM', 'psc', 'AP', 'KB', 'GC', 'SAM', 'TGM', 'TFP', 'GLC', 'OCL', 'PTNT', 'tci', 'oa', 'CAS', 'DA', 'FS', 'ca', 'FFH', 'CM', 'oc', 'SP', 'BAUPOST', 'LMM', 'WP', 'AC', 'TA', 'DAV', 'GR', 'SSHFX', 'EC', 'PI', 'AKO', 'FE', 'pcm', 'abc', 'RVC', 'ARFFX', 'CAAPX', 'T', 'JIM', 'cc', 'MKL', 'YAM', 'LT', 'WVALX', 'TWEBX', 'MPGFX', 'DODGX', 'MAVFX', 'pzfvx', 'FPACX', 'FPPTX', 'aq', 'OFALX']


## 2. 각 기업마다의 activity 부분

In [ ]:
# 각 기업마다의 activity 부분을 크롤링해주는 create_activity 함수 생성 및 사용

import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

# 폴더 경로 설정
folder_path = './f13_activity'
os.makedirs(folder_path, exist_ok=True)  # 폴더가 없으면 생성

def create_activity(who):
    quarter = []
    stock = []
    activity = []
    share_change = []
    percent_change = []

    for i in range(10):  # 10개의 페이지를 탐색
        with requests.Session() as session:
            session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
            url = f'https://www.dataroma.com/m/m_activity.php?m={who}&typ=a&L={i+1}'
            response = session.get(url)
            if response.status_code != 200:
                print(f"Failed to retrieve data: {response.status_code}")
                continue
            soup = BeautifulSoup(response.text, 'html.parser')

            # HTML 파싱하여 필요한 데이터 추출
            quarters = soup.find_all('tr', class_='q_chg')
            for quarter_tag in quarters:
                quarter_year = quarter_tag.text.strip()
                year = int(quarter_year.split()[-1])
                if year < 2023:
                    break  # 2023년 이전 데이터는 무시
                rows = quarter_tag.find_next_siblings('td')
                for j in range(0, len(rows), 5):
                    quarter.append(quarter_year)
                    stock.append(rows[j+1].text.strip().replace('\n', ' '))
                    activity.append(rows[j+2].text.strip())
                    share_change.append(rows[j+3].text.strip())
                    percent_change.append(rows[j+4].text.strip())

    df_activity = pd.DataFrame({
        'Quarter': quarter,
        'Stock': stock,
        'Activity': activity,
        'Share change': share_change,
        '% change to portfolio': percent_change
    })


    # 파일 경로 설정 및 저장
    csv_filename = f'{folder_path}/df_activity_{who}.csv'
    df_activity.to_csv(csv_filename, index=False)
    print(f"Saved to {csv_filename}")

    return df_activity

In [ ]:
'''
# 테스트 - SAM
df_holdings_SAM = create_activity('SAM')
df_holdings_SAM
'''

"\n# 테스트 - SAM\ndf_holdings_SAM = create_activity('SAM')\ndf_holdings_SAM\n"

In [ ]:
# 각 대형투자자별 activity csv파일 생성
for i in identifiers:
    df = create_activity(i)
    print(df)

Failed to retrieve data: 500
Saved to ./f13_activity/df_activity_hcmax.csv
      Quarter                              Stock       Activity Share change  \
0    Q3  2024                 INTC - Intel Corp.     Add 32.32%       44,700   
1    Q3  2024             EL - Estee Lauder Cos.     Add 31.82%       14,000   
2    Q3  2024  WBD - Warner Bros. Discovery Inc.     Add 26.76%      170,600   
3    Q3  2024              DIS - Walt Disney Co.     Add 25.00%       13,000   
4    Q3  2024       ZBH - Zimmer Biomet Holdings     Add 20.86%        9,200   
..        ...                                ...            ...          ...   
370  Q1  2023  WBD - Warner Bros. Discovery Inc.  Reduce 33.05%      178,300   
371  Q1  2023                    RTX - RTX Corp.   Sell 100.00%       55,000   
372  Q1  2023  WBD - Warner Bros. Discovery Inc.     Add 50.43%      180,866   
373  Q1  2023                 INTC - Intel Corp.     Add 15.91%       23,000   
374  Q1  2023              DIS - Walt Disney 

## 3. 각 기업마다의 holdings 부분

In [ ]:
# 각 기업마다의 holdings 부분을 크롤링해주는 create_holdings 함수 생성 및 사용

import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

# 폴더 경로 설정
folder_path = './f13_holdings'
os.makedirs(folder_path, exist_ok=True)  # 폴더가 없으면 생성

def create_holdings(who):
    # 빈 리스트를 생성하여 데이터를 저장
    stock = []
    portfolio_percentage = []
    recent_activity = []
    shares = []
    reported_price = []
    value = []
    current_price = []
    price_change = []
    week_low = []
    week_high = []

    for i in range(2):  # 2개의 페이지를 탐색
        with requests.Session() as session:
            session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
            url = f'https://www.dataroma.com/m/holdings.php?m={who}&L={i+1}'
            response = session.get(url)
            if response.status_code != 200:
                print(f"Failed to retrieve data: {response.status_code}")
                continue
            soup = BeautifulSoup(response.text, 'html.parser')


        # HTML 파싱하여 필요한 데이터 추출
        rows = soup.find_all('tr')[1:]  # 첫 번째 행은 헤더이므로 제외

        for row in rows: #각 행에 대해 반복
            cells = row.find_all('td')
            if len(cells) > 11:
                stock.append(cells[1].text.strip())  # Stock
                portfolio_percentage.append(cells[2].text.strip())  # % of Portfolio
                recent_activity.append(cells[3].text.strip())  # Recent Activity
                shares.append(cells[4].text.strip())  # Shares
                reported_price.append(cells[5].text.strip())  # Reported Price
                value.append(cells[6].text.strip())  # Value
                current_price.append(cells[8].text.strip())  # Current Price
                price_change.append(cells[9].text.strip())  # +/-
                week_low.append(cells[10].text.strip())  # 52 Week Low
                week_high.append(cells[11].text.strip())  # 52 Week High


    # 데이터프레임 생성
    df_holdings = pd.DataFrame({
        'Stock': stock,
        '% of Portfolio': portfolio_percentage,
        'Recent Activity': recent_activity,
        'Shares': shares,
        'Reported Price': reported_price,
        'Value': value,
        'Current Price': current_price,
        '+/- Reported Price': price_change,
        '52 Week Low': week_low,
        '52 Week High': week_high
    })

    # CSV 파일로 저장
    csv_filename = f'{folder_path}/df_holdings_{who}.csv'
    df_holdings.to_csv(csv_filename, index=False)
    print(f"Saved to {csv_filename}")

    return df_holdings

In [ ]:
'''
# 테스트 - SAM
df_holdings_SAM = create_holdings('SAM')
df_holdings_SAM
'''

"\n# 테스트 - SAM\ndf_holdings_SAM = create_holdings('SAM')\ndf_holdings_SAM\n"

In [ ]:
# 각 대형투자자별 activity csv파일 생성
for i in identifiers:
    df = create_holdings(i)
    print(df)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
20     13,930        $269.06   $3,748,000       $238.31            -11.43%   
21      4,330        $849.88   $3,680,000       $909.10              6.97%   
22    214,561         $13.49   $2,894,000         $9.35            -30.69%   
23     12,650        $210.59   $2,664,000       $229.54              9.00%   
24     33,169         $75.37   $2,500,000        $82.45              9.39%   
25     12,947        $158.96   $2,058,000       $189.28             19.07%   
26     22,425         $63.63   $1,427,000        $69.57              9.34%   
27      4,518        $231.52   $1,046,000       $271.42             17.23%   
28      6,036        $156.39     $944,000       $149.65             -4.31%   
29     16,487         $46.89     $773,000        $42.09            -10.24%   
30     27,072         $28.59     $774,000        $34.74             21.51%   
31      4,138        $171.58     $710,000       $266.60             55.38%   
32     23,869         $18.06

#네이버 해외뉴스 크롤링

In [ ]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver
!pip install webdriver-manager # install the missing module

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 1.5 MB/s eta 0:00:00
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Ign:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,159 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages

In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
from datetime import datetime, timedelta  # 날짜 계산을 위한 라이브러리

# ChromeDriver 경로 설정
#chrome_driver_path = "C:/Users/yywls/Documents/Untitled Folder/chromedriver.exe"



# ChromeDriver 경로 설정
chrome_driver_path = "C:/Users/chromedriver-win64/chromedriver.exe"  # 본인의 chromedriver 경로로 변경

# WebDriver 설정
service = Service(chrome_driver_path)

# Chrome 옵션 설정
options = Options()

# WebDriver 설정
driver = webdriver.Chrome(service=service, options=options)

# 페이지가 로드될 때까지 대기
def wait_for_element(locator):
    WebDriverWait(driver, 15).until(EC.presence_of_element_located(locator))

# 뉴스 데이터를 추출하는 함수
def scrape_news():
    news_items = driver.find_elements(By.CLASS_NAME, 'articleSubject')
    news_data = []
    for item in news_items:
        title = item.text
        link = item.find_element(By.TAG_NAME, 'a').get_attribute('href')
        news_data.append({'title': title, 'link': link})
    return news_data

# 특정 날짜에서 모든 페이지를 스크랩하는 함수
def scrape_all_pages():
    all_news = []  # 모든 페이지의 뉴스 데이터를 저장할 리스트
    page_num = 2  # 첫 번째 페이지는 이미 스크랩했으므로 2페이지부터 시작

    while True:
        print(f"페이지 {page_num - 1} 스크랩 중...")

        # 현재 페이지의 모든 뉴스 제목과 링크를 스크랩
        try:
            news_data = scrape_news()  # 현재 페이지의 뉴스 스크랩
            print(f"총 {len(news_data)}개의 뉴스 항목이 발견되었습니다.")
            all_news.extend(news_data)  # 현재 페이지 뉴스 데이터 리스트에 추가
        except:
            print("뉴스 항목을 스크랩하는 데 문제가 발생했습니다.")
            break

        # 다음 페이지로 이동 (페이지 번호로 찾기)
        next_page_button = find_next_page(page_num)
        if next_page_button:
            next_page_button.click()
            wait_for_element((By.CLASS_NAME, 'articleSubject'))  # 페이지가 로드될 때까지 대기
            time.sleep(3)  # 페이지 로드 시간 충분히 대기
            page_num += 1
        else:
            print(f"마지막 페이지 ({page_num - 1}) 입니다.\n")
            break

    return all_news

# 날짜별로 URL을 동적으로 변경하여 뉴스 스크래핑 (특정 날짜 범위)
def scrape_news_by_date_range(start_date, end_date):
    # 스크래핑할 URL 기본 형식
    base_url = 'https://finance.naver.com/news/news_list.naver?mode=LSS3D&section_id=101&section_id2=258&section_id3=403&date={}'

    # 시작 날짜와 종료 날짜를 datetime 객체로 변환
    start_date = datetime.strptime(start_date, '%Y%m%d')
    end_date = datetime.strptime(end_date, '%Y%m%d')

    # 날짜 범위를 거꾸로 진행하기 위해 timedelta를 사용하여 날짜를 감소시킴
    current_date = start_date
    scraped_data = []

    # 종료 날짜가 될 때까지 루프 실행
    while current_date >= end_date:
        # 날짜를 'YYYYMMDD' 형식으로 변환
        date_str = current_date.strftime('%Y%m%d')  # 예: 2024년 9월 27일 -> '20240927'

        # 날짜별 URL 설정
        target_url = base_url.format(date_str)
        print(f"접속 중: {target_url}")

        # 해당 날짜의 페이지로 이동
        driver.get(target_url)

        # 페이지 로드 대기
        try:
            wait_for_element((By.CLASS_NAME, 'articleSubject'))
            print(f"뉴스 목록이 로드되었습니다: {date_str}")
        except:
            print(f"뉴스 목록 로드 실패: {date_str}")
            current_date -= timedelta(days=1)  # 다음 날짜로 이동
            continue

        # 해당 날짜의 모든 페이지에서 뉴스 추출
        all_news = scrape_all_pages()

        # 각 뉴스 제목을 클릭하고 본문 내용을 가져오기
        for idx, news in enumerate(all_news):
            print(f"Title {idx + 1}: {news['title']}")

            # 뉴스 링크로 이동
            driver.get(news['link'])

            # 본문 내용이 있는 페이지로 이동할 때까지 대기
            try:
                wait_for_element((By.TAG_NAME, 'body'))
                print("뉴스 본문이 로드되었습니다.")
            except:
                print("뉴스 본문 로딩에 실패했습니다.")
                continue

            # 페이지 소스 가져오기
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')

            # 본문 내용 추출 (여기서 div, p 태그 또는 다른 클래스명으로 변경하여 시도 가능)
            content = soup.find('div', {'id': 'newsct_article'})
            if not content:
                content = soup.find('div', {'class': 'news_end'})

            # 본문 내용이 있는 경우만 저장 (출력하지 않고 저장만 진행)
            if content:
                news_content = content.get_text(strip=True)
                # 날짜, 제목, 본문 데이터를 리스트에 추가
                scraped_data.append({'날짜': date_str, '제목': news['title'], '본문내용': news_content})
            else:
                print(f"Content {idx + 1}: 본문을 찾지 못했습니다.")

            # 뉴스 본문 출력하지 않음
            print("-" * 80)

        # 다음 날짜로 이동 (하루씩 감소)
        current_date -= timedelta(days=1)

    # pandas DataFrame으로 변환
    news_df = pd.DataFrame(scraped_data)

    # 결과 확인 (출력하지 않음)
    return news_df

# 스크래핑 시작 (예: 2023년 9월 27일부터 2024년 9월 26일까지)
scraped_news_df = scrape_news_by_date_range(start_date='20230927', end_date='20240926')

# 결과 데이터프레임 저장 (저장만 하고 출력하지 않음)
scraped_news_df.to_csv('scraped_news_0927_0920.csv', index=False)

# 드라이버 종료
driver.quit()


NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


#뉴스 감성분석

In [ ]:
# 기본 라이브러리 설치
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 한글폰트 사용을 위해 설치
# 아래 모듈을 설치하고 불러오면 별도의 한글폰트 설정이 필요 없습니다.
!pip install koreanize-matplotlib

import koreanize_matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 45.6 MB/s eta 0:00:00


In [ ]:
# nlp 라이브러리 설치
from nltk.corpus import stopwords  # 불용어
import nltk
from tensorflow.keras.preprocessing.text import text_to_word_sequence  # 단어토큰화
from nltk.stem import WordNetLemmatizer  # 표제어 추출
from collections import defaultdict      # 딕셔너리 초기값 설정
from collections import  Counter         # 분포 시각화
from sklearn.feature_extraction.text import CountVectorizer    # ngram

from tensorflow.keras.preprocessing.text import Tokenizer   # 전처리->역토큰화 후 다시 토큰화
from tensorflow.keras.utils import to_categorical   # one-hot encoding
from sklearn.feature_extraction.text import CountVectorizer   #count vectorization
from sklearn.feature_extraction.text import TfidfVectorizer  #tfidf
from tensorflow.keras.preprocessing.sequence import pad_sequences  # padding
#import re
import gensim   # word embedding

In [ ]:
# modeling 및 embedding 층 쌓기
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten, Input, SpatialDropout1D, Dropout


# col 생략 없이 출력
pd.set_option('display.max_columns', None)
# row 생략 없이 출력
pd.set_option('display.max_rows', 20)
# col 최대 너비 200
pd.set_option('max_colwidth', 200)

In [ ]:
# 데이터 불러오기 - file_path 이용방법
file_path = '/content/drive/MyDrive/Colab Notebooks/news_Q2.csv'
data = pd.read_csv(file_path)
data.head()

,날짜,제목,본문내용
0,20240630,"""이젠 떠날 시간""…토론 참패 바이든, 후보 교체론 후폭풍","지지층, 바이든에 사퇴 압박대체 후보로 해리스, 뉴섬 등 거론바이든은 완주 의사 밝혀…""영부인에 달려""오는 11월 미국 대선을 앞두고 첫TV토론에서 조 바이든 대통령이 참패하면서 진보 진영에서 후보 교체 목소리가 커지고 있다. 바이든 대통령은 뉴욕주 모금 행사에서 사퇴 요구를 일축했지만, 8월 민주당 전당대회까지 후보 교체 논란은 수그러들지 않을 전..."
1,20240630,뜨거운 비만약 테마주…후발주자에 관심,'투톱' 사상 최고가일라이릴리·노보노디스크기대 넘어 실적으로 증명후발 주자론 암젠 주목월 1회 투약으로 개발하반기 2상 결과에 주목비만 치료제 테마를 앞세워 상반기 글로벌 증시를 달군 바이오주가 하반기에도 강세를 이어갈 것이란 전망이 나온다. 비만약 수요가 빠르게 늘고 있는 만큼 상승 추세가 당분간 계속될 것이란 관측이다. 증권가에서는 새 비만 치료...
2,20240630,"상하이 증시, 美·中 갈등 완화 기대…중국 증시 '사자'",지난주 마지막 거래일인 28일 중국 증시는 미·중 갈등 완화에 대한 기대감이 반영되면서 상승 마감했다. 이날 상하이종합지수와 선전종합지수는 전일 대비 각각 0.73%. 0.25% 올랐다. 홍콩 항셍지수도 전장 대비 0.01% 소폭 상승했다.조 바이든 미국 대통령과 도널드 트럼프 전 미국 대통령의 대선TV토론이 끝나자 중국 증시에서 매수세가 강해졌다....
3,20240630,"뉴욕 증시, 美 6월 실업률 4% 넘으면 금리인하 '탄력'","이번 주(1~5일) 뉴욕증시는 고용 지표 발표를 앞두고 있다. 미국 노동부의 비농업 고용 보고서, 민간 고용 보고서, 구인·구직 보고서 등이 공개된다. 현지시간 기준으로 △2일엔 5월 구인·이직 보고서(JOLTs) △3일엔 6월ADP고용 보고서가 나온다. 5일엔 6월 비농업 부문 신규 고용·실업률을 확인할 수 있다.최근 인플레이션이 다시 둔화하는 추..."
4,20240630,정치적 불확실성 극복한 인도·극우 득세 우려되는 유럽···정치 이벤트에 해외펀드 수익률 갈..,펀드 1달 수익률 인도 5.06%vs. 유럽-1.90%총선 결과 ‘모디 3기’ 개막한 인도는 승승장구유럽의회 선거는 강경우파가 3당 차지 ‘이변’30일 프랑스 총선도 극우·극좌가 1·2당 꿰찰듯<그림=챗GPT>‘정치의 해’라고 할만큼 올해 주요 국가에서 총선과 대선 등 굵직한 선거가 잇따르는 가운데 선거결과에 따라 각국에 투자하는 펀드 수익률이 극과...


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4581 entries, 0 to 4580
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   날짜      4581 non-null   int64 
 1   제목      4581 non-null   object
 2   본문내용    4555 non-null   object
dtypes: int64(1), object(2)
memory usage: 107.5+ KB


In [ ]:
data.isnull().sum()

,0
날짜,0
제목,0
본문내용,26


In [ ]:
data[data['날짜']==20240627]

,날짜,제목,본문내용
121,20240627,"뉴욕증시, 대선 첫 토론회 날…경제공약·PCE 기대하며 상승 출발","뉴욕증권거래소(로이터=연합뉴스 자료사진)(뉴욕=연합뉴스) 김 현 연합인포맥스 통신원 = 뉴욕증시는 미국 대선의 민주·공화 양당 후보인 조 바이든 대통령과 도널드 트럼프 전 대통령의 첫 토론회가 열리는 날, 새로 나온 경제 지표에 주목하고 참신한 경제 공약을 기대하며 3대 지수 모두 상승세로 출발했다.27일(현지시간) 뉴욕증권거래소(NYSE)에서 오전..."
122,20240627,"중국 지커, 러시아 EV 시장 점유율 확대…1년새 8천대 판매","중국의 전기자동차(EV) 제조업체 지커(Zeekr)가 러시아에서 빠르게 성장하고 있다.27일(현지시간) 로이터에 따르면 러시아 분석 기관인 오토스타트(Autostat)는 2023년 5월부터 2024년 4월까지 러시아에서는 20,500대 이상의 새로운EV가 판매됐으며, 이는 전년 대비 약 350% 증가한 수치라고 밝혔다.이 중 중국 브랜드가 판매량의 ..."
123,20240627,"소프트뱅크그룹, 템퍼스 AI와 AI 헬스케어 합작법인 설립",일본 소프트뱅크그룹은 템퍼스AI와 합작법인을 출시한다고 27일(현지시간) 로이터가 밝혔다.합작법인은 개인 의료 데이터를 인공지능(AI)으로 분석해 치료법 추천을 제시하는 것을 목표로 한다.이는 소프트뱅크가 몇 년 만에 투자 활동 속도를 높이면서 최근 발표한 일련의AI투자 중 최신 소식이다.소프트뱅크는 템퍼스AI가 6월 나스닥에 상장되기 전인 4월 ...
124,20240627,"뉴욕증시, 고용지표 둔화 소화하며 상승…국채금리 하락","美 계속 실업수당 청구, 31개월 만에 최고1분기GDP확정치는 1.4%…0.1%P ↑5월PCE물가 28일 발표미국 뉴욕증시의 3대 지수는 27일(현지시간) 장 초반 강보합세다. 기업들의 실적이 엇갈린 가운데 고용시장 둔화 조짐이 확인되면서 지수가 소폭 상승하고 있다. 투자자들의 시선은 28일 발표되는 개인소비지출(PCE) 물가지수로 쏠린다.[이미지출..."
125,20240627,"美 계속 실업수당 청구, 31개월 만에 최고",지난주 2주 이상 실업수당을 신청하는 계속 실업수당 청구건수가 2년 7개월 만에 가장 높은 수준을 기록한 것으로 나타났다.27일(현지시간) 미국 노동부에 따르면 지난주(6월16~22일) 신규 실업수당 청구 건수는 23만3000건으로 집계됐다. 전문가 전망치(23만6000건)를 소폭 하회하는 수준이다. 한 주 전 23만9000건(수정치) 보다는 감소했...
...,...,...,...
181,20240627,"美 유권자 74% ""첫 대선 TV 토론, 선거 결과에 중요""","AP통신·시카고대 여론연구센터 조사가상대결 지지율, 트럼프 48%·바이든 45%미국 유권자 10명 중 7명 이상은 27일(현지시간) 첫 대선TV토론이 선거 결과를 좌우할 주요 분수령이 될 것이라 보는 것으로 나타났다.26일AP통신과 시카고대 여론연구센터(NORC)에 따르면 지난 20~24일 유권자 1088명을 대상으로 여론조사를 실시한 결과 응답자의..."
182,20240627,"[속보]뉴욕증시, 상승 마감…나스닥 0.49% ↑","미국 뉴욕 증시의 3대 지수는 26일(현지시간) 상승세로 마감했다.이날 뉴욕증권거래소(NYSE)에서 다우존스30산업평균지수는 전 거래일보다 0.04% 올라 거래를 마쳤다. 대형주 중심의S&P500지수는 0.16%, 기술주 중심의 나스닥 지수는 0.49% 올라 장을 마감했다."
183,20240627,"美 투자은행 10곳 중 6곳 ""Fed, 연내 2회 이상 금리 인하 전망""",한은 뉴욕사무소 주요IB전망 집계9월까지 첫 금리 인하 착수 예상올해 성장률 2.4%·물가상승률 2.8% 예상미국 월가 주요 투자은행(IB) 10곳 중 6곳은 미 연방준비제도(Fed)가 올해 금리를 2회 이상 인하할 것으로 내다봤다. 금리 인하 전망 시점으로는 오는 9월이 가장 유력했다.IB대부분은 올해 미국 경제가 2% 초반의 성장률을 기록해 노랜...
184,20240627,최대 전자상거래 아마존도 시총 2조달러 '터치'…美 역대 5번째,아마존 로고[로이터 연합뉴스 자료사진. 재판매 및DB금지](샌프란시스코=연합뉴스) 김태종 특파원 = 세계 최대 전자상거래 업체 아마존의 시가총액이 26일(현지시간) 장중 처음으로 2조 달러를 '터치'했다.미 동부 시간 기준 이날 낮 12시 50분(서부 시간 오전 9시 50분) 아마존 주가는 3.43% 상승한 192.73달러(26만8천376원)에 거래...


In [ ]:
data[data['본문내용'].isnull()==True]
# 대부분 속보인 경우, 동일 날짜에 비슷한 뉴스가 많으므로 제거해도 무방하다 판단

,날짜,제목,본문내용
168,20240627,"[1보] 네이버웹툰, 나스닥 공모가격 주당 21달러…희망가 상단 결정",NaN
578,20240618,[속보]美 5월 소매판매 전월比 0.1% 증가…예상 하회,NaN
869,20240613,"[속보]파월 ""주거비, 금리에 영향 미쳐""",NaN
870,20240613,"[속보]파월 ""고용시장, 점진적 냉각…점차 수급 균형""",NaN
871,20240613,"[속보]파월 ""5월 CPI 보고서 진전…정책 완화엔 충분치 않아""",NaN
...,...,...,...
2941,20240502,"[속보]파월 ""인플레 여전히 너무 높아…인플레 하락 진전 장담 못해""",NaN
2944,20240502,"[속보]美 Fed ""최근 몇 달간 인플레 2% 둔화 위한 추가 진전 없어""",NaN
2947,20240502,"[속보]美 Fed, 6연속 기준금리 동결…연 5.25~5.5% 유지",NaN
3151,20240426,[속보]美 3월 PCE 물가지수 전년대비 2.7%↑…예상치 상회,NaN


In [ ]:
data = data.dropna() # 결측치 제거

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4555 entries, 0 to 4580
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   날짜      4555 non-null   int64 
 1   제목      4555 non-null   object
 2   본문내용    4555 non-null   object
dtypes: int64(1), object(2)
memory usage: 142.3+ KB


#특수문자 제거 및 숫자 제거
텍스트에서 의미없는 특수문자나 숫자를 제거

In [ ]:
texts = data['본문내용']
texts

,본문내용
0,"지지층, 바이든에 사퇴 압박대체 후보로 해리스, 뉴섬 등 거론바이든은 완주 의사 밝혀…""영부인에 달려""오는 11월 미국 대선을 앞두고 첫TV토론에서 조 바이든 대통령이 참패하면서 진보 진영에서 후보 교체 목소리가 커지고 있다. 바이든 대통령은 뉴욕주 모금 행사에서 사퇴 요구를 일축했지만, 8월 민주당 전당대회까지 후보 교체 논란은 수그러들지 않을 전..."
1,'투톱' 사상 최고가일라이릴리·노보노디스크기대 넘어 실적으로 증명후발 주자론 암젠 주목월 1회 투약으로 개발하반기 2상 결과에 주목비만 치료제 테마를 앞세워 상반기 글로벌 증시를 달군 바이오주가 하반기에도 강세를 이어갈 것이란 전망이 나온다. 비만약 수요가 빠르게 늘고 있는 만큼 상승 추세가 당분간 계속될 것이란 관측이다. 증권가에서는 새 비만 치료...
2,지난주 마지막 거래일인 28일 중국 증시는 미·중 갈등 완화에 대한 기대감이 반영되면서 상승 마감했다. 이날 상하이종합지수와 선전종합지수는 전일 대비 각각 0.73%. 0.25% 올랐다. 홍콩 항셍지수도 전장 대비 0.01% 소폭 상승했다.조 바이든 미국 대통령과 도널드 트럼프 전 미국 대통령의 대선TV토론이 끝나자 중국 증시에서 매수세가 강해졌다....
3,"이번 주(1~5일) 뉴욕증시는 고용 지표 발표를 앞두고 있다. 미국 노동부의 비농업 고용 보고서, 민간 고용 보고서, 구인·구직 보고서 등이 공개된다. 현지시간 기준으로 △2일엔 5월 구인·이직 보고서(JOLTs) △3일엔 6월ADP고용 보고서가 나온다. 5일엔 6월 비농업 부문 신규 고용·실업률을 확인할 수 있다.최근 인플레이션이 다시 둔화하는 추..."
4,펀드 1달 수익률 인도 5.06%vs. 유럽-1.90%총선 결과 ‘모디 3기’ 개막한 인도는 승승장구유럽의회 선거는 강경우파가 3당 차지 ‘이변’30일 프랑스 총선도 극우·극좌가 1·2당 꿰찰듯<그림=챗GPT>‘정치의 해’라고 할만큼 올해 주요 국가에서 총선과 대선 등 굵직한 선거가 잇따르는 가운데 선거결과에 따라 각국에 투자하는 펀드 수익률이 극과...
...,...
4576,"[서울경제]29일(현지시간) 뉴욕증시는 성금요일로 휴장 했다.이날 아시아 증시는 상승세를 보였다.일본 닛케이 지수는 201.37포인트(0.50%) 뛴 4만 369.44에, 중국 상하이 지수는 30.50포인트(1.01%) 오른 3041.17에 장을 마감했다.대만 자취엔 지수는 전일보다 147.90포인트(0.73)% 상승한 2만 294.45로 거래를 마쳤다."
4577,"JP모건, 선진국 근원물가 분석작년 하반기 3%→올해 1분기 3.5%미국, 유럽 등 주요 선진국의 올해 1분기 근원 물가 상승률이 확대됐다는 분석이 나왔다. 주요국 인플레이션이 정점 대비로는 크게 둔화했으나 하락세가 멈추거나 소폭 반등하면서 연내 금리 인하를 앞두고 각국 중앙은행이 '라스트 마일(lastmile·목표에 이르기 직전 최종 구간)' 리스..."
4578,[글로벌시장지표/ 한국시간 기준 4월 1일 오전 6시 현재][미국증시 마감시황]뉴욕증시는 이번주 2분기를 시작한다.투자자들은 이번주 발표되는 고용지표들에 초점을 맞출 전망이다.특히 8일(현지시간) 예정된 3월 고용동향은 미국 연방준비제도(연준)의 금리인하 시기를 좌우할 핵심 변수다.투자자들은 아울러 2일 발표 예정인 테슬라의 1분기 출하통계에도 촉각...
4579,"폭스콘, 멕시코서AI서버 생산美 아마존·구글·MS·엔비디아 등 공급대만 폭스콘이 멕시코에서 인공지능(AI) 서버 생산을 확대한다. 반도체,AI등 첨단기술 분야에서 미·중 경쟁이 가열되면서 미국 주요 기업들이 협력사에 생산기지를 중국에서 멕시코로 이전하라고 요구하면서다. 미국의 대 중국 견제가 가속화되는 상황에서 '니어쇼어링(인접국으로 생산기지)' 전..."


In [ ]:
import re

def clean_text(text):
    # 특수문자, 숫자 제거 (정규표현식 사용)
    text = re.sub(r'[^가-힣\s]', '', text)  # 한글과 공백을 제외한 모든 문자 제거
    return text

In [ ]:
retexts = []
for i in texts:
    retexts.append(clean_text(i))
retexts[:2]

['지지층 바이든에 사퇴 압박대체 후보로 해리스 뉴섬 등 거론바이든은 완주 의사 밝혀영부인에 달려오는 월 미국 대선을 앞두고 첫토론에서 조 바이든 대통령이 참패하면서 진보 진영에서 후보 교체 목소리가 커지고 있다 바이든 대통령은 뉴욕주 모금 행사에서 사퇴 요구를 일축했지만 월 민주당 전당대회까지 후보 교체 논란은 수그러들지 않을 전망이다일현지시간 주요 외신에 따르면 워터게이트 사건 특종 기자인 밥 우드워드는 전날방송에 출연해 바이든 대통령의 토론이 너무 나쁘고 끔찍했다며 이는 단지 바이든 대통령과 민주당에만 정치적 수소폭탄인 것이 아니다라고 평가했다 그러면서 후보 교체 요구는 피할 수 없다고 강조했다뉴욕타임스 대표 칼럼니스트인 토머스 프리드먼은 바이든 대통령의 토론 모습을 보고 흐느꼈다며 품위를 지키고 무대를 떠나야 한다고 칼럼에 썼다바이든 대통령의 어린 시절 친구이자 지지자인 작가 제이 파라니는 미국방송에 조에게 이제 떠날 시간이다라는 제목의 서한을 보내 대통령직 사퇴를 촉구했다앞서 바이든 대통령은 일 첫 대선토론에서 트럼프 전 대통령과 달리 말을 더듬거나 허공을 응시하는 등의 모습을 보이는 등 고령 리스크를 부각 후보 교체론을 점화했다 여론조사기관 모닝컨설트가토론 후 유권자 명을 대상으로 실시한 조사에 따르면 응답자 는 바이든 대통령이 후보에서 교체돼야 한다고 답했다 바이든 대통령을 대체할 후보로는 카멀라 해리스 부통령 개빈 뉴섬 캘리포니아 주지사 조시 셔피로 펜실베이니아 주지사 그레첸 휘트머 미시간 주지사프리츠커 일리노이 주지사 등이 거론된다하지만 바이든 대통령은 완주 의사를 다지고 있다 그는 일 뉴욕주 햄프턴에서 열린 선거 자금 모금 행사에서 승리를 자신하며 지지를 호소했다 바이든 대통령은 명 참석자 앞에서 토론에 대한 우려를 이해한다 나는 멋진 밤을 보내지 못했다면서도 내가 승리할 거라 믿지 않았다면 대선에 출마하지 않았을 것이라고 강조했다 버락 오바마 전 대통령 빌 클린턴 전 대통령 낸시 펠로시 전 하원의장 등 민주당 주요 인사들도 바이든 대통령에 

### 형태소 분석+ 어간 추출 및 표제어 추출 + 불필요한 조사, 접속사 제거
형태소 분석: 한국어는 어미 변화, 조사 결합 등이 많이 일어나기 때문에, 단어를 기본형(어근)으로 변환하는 형태소 분석이 필요합니다. 이를 통해 단어의 기본형을 추출하고 명사, 동사, 형용사 등을 분리할 수 있습니다.

형태소 분석과 유사하게, 동사나 형용사의 어미를 제거하고 어간을 추출하는 작업입니다. 이는 단어의 기본 형태를 유지하기 위한 방법으로, '먹다', '먹는다', '먹었다'를 모두 '먹다'로 변환합니다.

In [ ]:
!pip install konlpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.6/488.6 kB 21.5 MB/s eta 0:00:00


In [ ]:
from konlpy.tag import Okt

okt = Okt()

def stemming(text):
    # 텍스트의 각 단어를 형태소 분석하고, 표제어 추출(stem=True) 수행
    stemmed_words = okt.pos(text, stem=True)

    # 조사(Josa)와 접속사(Conjunction)를 제거
    filtered_words = [word for word, pos in stemmed_words if pos not in ['Josa', 'Conjunction']]

    return filtered_words

In [ ]:
retexts2 = []
for i in retexts:
    retexts2.append(stemming(i))
retexts2[:2]

[['지',
  '지층',
  '바이든',
  '사퇴',
  '압박',
  '대체',
  '후보',
  '해리스',
  '뉴섬',
  '등',
  '거론',
  '바이든',
  '완주',
  '의사',
  '밝히다',
  '영부인',
  '달려오다',
  '월',
  '미국',
  '대선',
  '앞두다',
  '첫',
  '토론',
  '조',
  '바이든',
  '대통령',
  '참패',
  '하다',
  '진보',
  '진영',
  '후보',
  '교체',
  '목소리',
  '커지다',
  '있다',
  '바이든',
  '대통령',
  '뉴욕주',
  '모금',
  '행사',
  '사퇴',
  '요구',
  '일',
  '축',
  '하다',
  '월',
  '민주당',
  '전당대회',
  '후보',
  '교체',
  '논란',
  '수',
  '그렇다',
  '들다',
  '않다',
  '전망',
  '일',
  '현',
  '지',
  '시간',
  '주요',
  '외신',
  '따르다',
  '워터게이트',
  '사건',
  '특종',
  '기자',
  '밥',
  '우드워드',
  '전날',
  '방송',
  '출연',
  '하다',
  '바이든',
  '대통령',
  '토론',
  '너무',
  '나쁘다',
  '끔찍하다',
  '이다',
  '단지',
  '바이든',
  '대통령',
  '민주당',
  '정치',
  '적',
  '수',
  '소',
  '폭탄',
  '것',
  '아니다',
  '평가',
  '하다',
  '그렇다',
  '후보',
  '교체',
  '요구',
  '피',
  '하다',
  '수',
  '없다',
  '강조',
  '하다',
  '뉴욕타임스',
  '대표',
  '칼럼니스트',
  '토머스',
  '프리드',
  '멀다',
  '바이든',
  '대통령',
  '토론',
  '모습',
  '보고',
  '흐',
  '느끼다',
  '품위',
  '지키다',
  '무대',
  '떠나다',
  '하다',
  '칼

추가적으로 불용어 제거

In [ ]:
f = open('stopwords_ko.txt','r')
stopwords = f.read()
stopwords=stopwords.split(' ')
stopwords

FileNotFoundError: [Errno 2] No such file or directory: 'stopwords_ko.txt'

In [ ]:
retexts3 = []
for i in retexts2:
    final_tokens = [word for word in i if word not in stopwords]
    retexts3.append(final_tokens)
retexts3[:2]

[['지층',
  '바이든',
  '사퇴',
  '압박',
  '대체',
  '후보',
  '해리스',
  '뉴섬',
  '거론',
  '바이든',
  '완주',
  '의사',
  '밝히다',
  '영부인',
  '달려오다',
  '미국',
  '대선',
  '앞두다',
  '첫',
  '토론',
  '조',
  '바이든',
  '대통령',
  '참패',
  '진보',
  '진영',
  '후보',
  '교체',
  '목소리',
  '커지다',
  '바이든',
  '대통령',
  '뉴욕주',
  '모금',
  '행사',
  '사퇴',
  '요구',
  '축',
  '민주당',
  '전당대회',
  '후보',
  '교체',
  '논란',
  '그렇다',
  '들다',
  '전망',
  '현',
  '주요',
  '외신',
  '따르다',
  '워터게이트',
  '사건',
  '특종',
  '기자',
  '밥',
  '우드워드',
  '전날',
  '방송',
  '출연',
  '바이든',
  '대통령',
  '토론',
  '너무',
  '나쁘다',
  '끔찍하다',
  '이다',
  '바이든',
  '대통령',
  '민주당',
  '정치',
  '소',
  '폭탄',
  '평가',
  '그렇다',
  '후보',
  '교체',
  '요구',
  '피',
  '강조',
  '뉴욕타임스',
  '대표',
  '칼럼니스트',
  '토머스',
  '프리드',
  '멀다',
  '바이든',
  '대통령',
  '토론',
  '모습',
  '보고',
  '흐',
  '느끼다',
  '품위',
  '지키다',
  '무대',
  '떠나다',
  '칼럼',
  '써다',
  '바이든',
  '대통령',
  '어리다',
  '시절',
  '친구',
  '이자',
  '지지자',
  '작가',
  '제이',
  '파라',
  '늘다',
  '미국방송',
  '조',
  '이제',
  '떠나다',
  '제목',
  '서한',
  '보내다',
  '대통령직',
  '사퇴',
  '촉구',


In [ ]:
# 문장으로 합치기
final_texts = []
for i in retexts3:
    final_texts.append(' '.join(i))
final_texts[:2]

['지층 바이든 사퇴 압박 대체 후보 해리스 뉴섬 거론 바이든 완주 의사 밝히다 영부인 달려오다 미국 대선 앞두다 첫 토론 조 바이든 대통령 참패 진보 진영 후보 교체 목소리 커지다 바이든 대통령 뉴욕주 모금 행사 사퇴 요구 축 민주당 전당대회 후보 교체 논란 그렇다 들다 전망 현 주요 외신 따르다 워터게이트 사건 특종 기자 밥 우드워드 전날 방송 출연 바이든 대통령 토론 너무 나쁘다 끔찍하다 이다 바이든 대통령 민주당 정치 소 폭탄 평가 그렇다 후보 교체 요구 피 강조 뉴욕타임스 대표 칼럼니스트 토머스 프리드 멀다 바이든 대통령 토론 모습 보고 흐 느끼다 품위 지키다 무대 떠나다 칼럼 써다 바이든 대통령 어리다 시절 친구 이자 지지자 작가 제이 파라 늘다 미국방송 조 이제 떠나다 제목 서한 보내다 대통령직 사퇴 촉구 앞서 바이든 대통령 첫 대선 토론 트럼프 대통령 달리 더듬다 허공 응시 모습 보이다 고령 리스크 부각 후보 교체 론 점화 여론조사 기관 모닝 컨설트 토론 유권자 대상 실시 조사 따르다 응답 늘다 바이든 대통령 후보 교체 돼다 답 바이든 대통령 대체 후보 카멀 해리스 부통령 개빈 뉴섬 캘리포니아 지사 좋다 셔피 펜실베이니아 지사 그레첸 휘트머 미시간 주지 사프리 츠커 일리노이 지사 거론 바이든 대통령 완주 의사 다지 뉴욕주 햄프턴 열리다 선거 자금 모금 행사 승리 지지 호소 바이든 대통령 참석자 토론 대한 우려 이해 다 멋지다 밤 보내다 못 승리 거 믿다 대선 출마 강조 버락 오바마 대통령 빌다 클린턴 대통령 낸시 펠 로시 하원 의장 민주당 주요 인사 바이든 대통령 대한 지지 의사 잇다 달 표명 엄호 가운데 바이든 대통령 주말 대통령 별장 캠프 데이비드 가족 보내다 자리 향후 계획 논의 전망 나오다 영부인 질 바이든 여사 후보 사퇴 관련 최종 결정 권 가지다 분석 미국방송 민주당 수뇌부 대통령 가족 상의 선거운동 계속 조기 끝내다 인지 결정 믿다 보도 방송 소식통 인용 바이든 궁극 영향력 가지다 유일하다 인물 영부인 면서 경로 변경 결정 경로 변경

In [ ]:
df = pd.DataFrame(final_texts, columns=['main_texts'])
df

,main_texts
0,지층 바이든 사퇴 압박 대체 후보 해리스 뉴섬 거론 바이든 완주 의사 밝히다 영부인 달려오다 미국 대선 앞두다 첫 토론 조 바이든 대통령 참패 진보 진영 후보 교체 목소리 커지다 바이든 대통령 뉴욕주 모금 행사 사퇴 요구 축 민주당 전당대회 후보 교체 논란 그렇다 들다 전망 현 주요 외신 따르다 워터게이트 사건 특종 기자 밥 우드워드 전날 방송 출연...
1,투톱 사상 최고 일라이 릴리 노 보노 디스크 기대 넘다 실적 증명 후발 주자 론 암 젠 주목 회 투약 개발 하반기 상 결과 주목 비만 치료 테마 앞세우다 상반기 글로벌 증시 달구다 바이오 주가 하반기 강세 이다 갈다 전망 나오다 비 수요 빠르다 늘 상승 추세 당분간 계속 관측 증권 가다 새 비만 치료 개발 속도 내다 발주 주목 미국 뉴욕증시 일라이 ...
2,지난주 마지막 거래 일인 중국 증시 밉다 갈등 완화 대한 기 대감 반영 상승 마감 날 상하이 종합 지수 선전 종합 지수 전일 대비 홍콩 항셍지수 전장 대비 소 폭 상승 조 바이든 미국 대통령 도널드 트럼프 미국 대통령 대선 토론 끝나다 중국 증시 매다 강하다 시장 중국 주요 이슈 다루다 호재 해석 바이든 트럼프 후보 경제 낙태 불법 이민 우크라이나 ...
3,주일 뉴욕증시 고용 지표 발표 앞두다 미국 노동부 비 농업 고용 보고서 민간 고용 보고서 구인 구직 보고서 공개 현지 기준 구인 이직 보고서 고용 보고서 나오다 비 농업 부문 신규 고용 실업률 확인 최근 인플레이션 둔화 추세 고용 시장 영향 받다 가능성 관심 쏠리다 미국 실업률 지난 이후 처음 넘다 금융시장 실업률 수준 추정 높다 실업률 미국 중앙은...
4,펀드 달 수익률 인도 유럽 총선 결과 모디 기 개막 인도 승승장구 유럽의회 선거 강경 우파 당 차지 변일 프랑스 총선 극우 극좌 당 꿰 차다 그림 챗 정치 해 올해 주요 국가 총선 대선 굵직하다 선거 잇따르다 가운데 선거 결과 따르다 각국 투자 펀드 수익률 극 극 갈리 선거 정치 불확실 성 해소 지역 우 상향 어가 반면 극단 주의 정 파의 득세 가능...
...,...
4550,서 울 경 현 뉴욕증시 성금요일 휴장 날 아시아 증시 상승세 보이다 일본 닛 케이 지수 포인트 뛰다 중국 상하이 지수 포인트 오른 장 마감 대만 자취 지수 전일 포인트 상승 거래 마치다
4551,모건 선진국 근원 물가 분석 작년 하반기 올해 분기 미국 유럽 주요 선진국 올해 분기 근원 물가 상 승률 확대 돼다 분석 나오다 주요 국 인플레이션 정점 대비 크게 둔화 하락 세 멈추다 소 폭 반등 연내 금리 인하 앞두다 각국 중앙은행 라스트 마일 목표 직전 최종 구간 리스크 직면 우려 제기 지난달 현 미국 투자 은행 모건 추산 주요 선진국 근원 물...
4552,글로벌 시장 지표 한국 기준 오전 시 현재 미국증시 마감 시 황 뉴욕증시 분기 시작 다 투자자 발표 고용 지표 초점 맞추다 전망 특히 현 예정 고용 동향 미국 연방 준비 제도 연 준 금리인하 시기 좌우 핵심 변수 투자자 아우르다 발표 예정 테슬라 분기 출하 통계 촉각 곤 세우다 전기차 수요 부진 테슬라 분기 출하 통계 테슬라 뿐 전기차 전반 주가 흐...
4553,폭스콘 멕시코 서버 생산 아마존 구글 엔비디아 공급 대만 폭스콘 멕시코 인공 지능 서버 생산 확대 다 반도체 첨단 기술 분야 밉다 경쟁 가열 미국 주요 기업 협력 생산 기지 중국 멕시코 전하 요구 서다 미국 대다 중국 견제 가속 화 상황 니 쇼 링 인접 국 생산 기지 전략 강화하다 기업 증가 지난달 현 월스트리트저널 따르다 폭스콘 지난 멕시코 서부 ...


#감성분석- koBert

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import torch

model_name="monologg/kobert"
#model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# GPU 사용 여부 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 감성 분석 함수 정의
def kobert_sentiment_analysis(text):
    # 입력 데이터 전처리
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    # 모델 예측 수행
    with torch.no_grad():
        outputs = model(**inputs)
    # 로짓(logits)을 소프트맥스로 변환하여 감성 점수 계산
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    # 가장 높은 감성 점수를 가진 클래스를 선택
    sentiment_class = torch.argmax(probs, dim=-1).item()
    sentiment_score = probs[0][sentiment_class].item()

    # 감성 클래스 매핑 (0: Negative, 1: Neutral, 2: Positive)
    sentiment_label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

    return sentiment_label[sentiment_class], sentiment_score

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at monologg/kobert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
df[['sentiment', 'sentiment_score']] = df['main_texts'].apply(
    lambda x: pd.Series(kobert_sentiment_analysis(x)))

In [ ]:
df

,main_texts,sentiment,sentiment_score
0,지층 바이든 사퇴 압박 대체 후보 해리스 뉴섬 거론 바이든 완주 의사 밝히다 영부인 달려오다 미국 대선 앞두다 첫 토론 조 바이든 대통령 참패 진보 진영 후보 교체 목소리 커지다 바이든 대통령 뉴욕주 모금 행사 사퇴 요구 축 민주당 전당대회 후보 교체 논란 그렇다 들다 전망 현 주요 외신 따르다 워터게이트 사건 특종 기자 밥 우드워드 전날 방송 출연...,Positive,0.359723
1,투톱 사상 최고 일라이 릴리 노 보노 디스크 기대 넘다 실적 증명 후발 주자 론 암 젠 주목 회 투약 개발 하반기 상 결과 주목 비만 치료 테마 앞세우다 상반기 글로벌 증시 달구다 바이오 주가 하반기 강세 이다 갈다 전망 나오다 비 수요 빠르다 늘 상승 추세 당분간 계속 관측 증권 가다 새 비만 치료 개발 속도 내다 발주 주목 미국 뉴욕증시 일라이 ...,Negative,0.371490
2,지난주 마지막 거래 일인 중국 증시 밉다 갈등 완화 대한 기 대감 반영 상승 마감 날 상하이 종합 지수 선전 종합 지수 전일 대비 홍콩 항셍지수 전장 대비 소 폭 상승 조 바이든 미국 대통령 도널드 트럼프 미국 대통령 대선 토론 끝나다 중국 증시 매다 강하다 시장 중국 주요 이슈 다루다 호재 해석 바이든 트럼프 후보 경제 낙태 불법 이민 우크라이나 ...,Negative,0.344473
3,주일 뉴욕증시 고용 지표 발표 앞두다 미국 노동부 비 농업 고용 보고서 민간 고용 보고서 구인 구직 보고서 공개 현지 기준 구인 이직 보고서 고용 보고서 나오다 비 농업 부문 신규 고용 실업률 확인 최근 인플레이션 둔화 추세 고용 시장 영향 받다 가능성 관심 쏠리다 미국 실업률 지난 이후 처음 넘다 금융시장 실업률 수준 추정 높다 실업률 미국 중앙은...,Positive,0.355032
4,펀드 달 수익률 인도 유럽 총선 결과 모디 기 개막 인도 승승장구 유럽의회 선거 강경 우파 당 차지 변일 프랑스 총선 극우 극좌 당 꿰 차다 그림 챗 정치 해 올해 주요 국가 총선 대선 굵직하다 선거 잇따르다 가운데 선거 결과 따르다 각국 투자 펀드 수익률 극 극 갈리 선거 정치 불확실 성 해소 지역 우 상향 어가 반면 극단 주의 정 파의 득세 가능...,Negative,0.366283
...,...,...,...
4550,서 울 경 현 뉴욕증시 성금요일 휴장 날 아시아 증시 상승세 보이다 일본 닛 케이 지수 포인트 뛰다 중국 상하이 지수 포인트 오른 장 마감 대만 자취 지수 전일 포인트 상승 거래 마치다,Negative,0.388989
4551,모건 선진국 근원 물가 분석 작년 하반기 올해 분기 미국 유럽 주요 선진국 올해 분기 근원 물가 상 승률 확대 돼다 분석 나오다 주요 국 인플레이션 정점 대비 크게 둔화 하락 세 멈추다 소 폭 반등 연내 금리 인하 앞두다 각국 중앙은행 라스트 마일 목표 직전 최종 구간 리스크 직면 우려 제기 지난달 현 미국 투자 은행 모건 추산 주요 선진국 근원 물...,Negative,0.360862
4552,글로벌 시장 지표 한국 기준 오전 시 현재 미국증시 마감 시 황 뉴욕증시 분기 시작 다 투자자 발표 고용 지표 초점 맞추다 전망 특히 현 예정 고용 동향 미국 연방 준비 제도 연 준 금리인하 시기 좌우 핵심 변수 투자자 아우르다 발표 예정 테슬라 분기 출하 통계 촉각 곤 세우다 전기차 수요 부진 테슬라 분기 출하 통계 테슬라 뿐 전기차 전반 주가 흐...,Negative,0.348821
4553,폭스콘 멕시코 서버 생산 아마존 구글 엔비디아 공급 대만 폭스콘 멕시코 인공 지능 서버 생산 확대 다 반도체 첨단 기술 분야 밉다 경쟁 가열 미국 주요 기업 협력 생산 기지 중국 멕시코 전하 요구 서다 미국 대다 중국 견제 가속 화 상황 니 쇼 링 인접 국 생산 기지 전략 강화하다 기업 증가 지난달 현 월스트리트저널 따르다 폭스콘 지난 멕시코 서부 ...,Negative,0.371545


In [ ]:
df['sentiment'].value_counts()

sentiment
Negative    3451
Positive     976
Neutral      128
Name: count, dtype: int64

#감성분석- KoFinBERT

In [ ]:
# KoFinBERT 모델 로드 (공개된 모델)
model_name = "snunlp/KR-FinBERT"  # 공개된 한국어 금융 BERT 모델
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# GPU 사용 여부 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 감성 분석 함수 정의
def kofinbert_sentiment_analysis(text):
    # 입력 데이터 전처리
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    # 모델 예측 수행
    with torch.no_grad():
        outputs = model(**inputs)
    # 로짓(logits)을 소프트맥스로 변환하여 감성 점수 계산
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    # 가장 높은 감성 점수를 가진 클래스를 선택
    sentiment_class = torch.argmax(probs, dim=-1).item()
    sentiment_score = probs[0][sentiment_class].item()

    # 감성 클래스 매핑 (0: Negative, 1: Neutral, 2: Positive)
    sentiment_label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

    return sentiment_label[sentiment_class], sentiment_score

## 개체명 인식 - NER

In [ ]:
import pandas as pd
from gliner import GLiNER

# GLiNER 모델 불러오기
model = GLiNER.from_pretrained("urchade/gliner_multi")

# GLiNER에서 사용할 라벨 (원하는 개체명 종류)
labels = ["Person", "Organization", "Product", "Term"]

# 개체명 추출 함수
def extract_entities(text, model, labels):
    entities = model.predict_entities(text, labels)
    # 추출된 엔티티들을 label별로 정리하여 딕셔너리로 반환
    entities_ref = {label: [] for label in labels}
    for entity in entities:
        entities_ref[entity['label']].append(entity['text'])
    return entities_ref

# 각 텍스트에서 개체명 추출 후 데이터프레임에 새로운 열 추가
df['entities'] = df['main_texts'].apply(lambda text: extract_entities(text, model, labels))

# 결과 출력
print(df)